In [ ]:
base_path = "/content"

In [ ]:
!sudo apt install tesseract-ocr
!pip install opencv-python numpy Pillow torch torchvision transformers pytesseract
from IPython.display import clear_output
clear_output()

In [ ]:
import requests
import os
import torch
import cv2
from google.colab.patches import cv2_imshow
import pytesseract
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from accelerate import Accelerator
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Grounding DINO



In [ ]:
model_id = "IDEA-Research/grounding-dino-tiny"
device = Accelerator().device
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/689M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

In [ ]:
def grounding_dino(image_url, text_labels, debug = False):
  image = Image.open(image_url).convert("RGB")
  inputs = processor(images=image, text=text_labels, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model(**inputs)

  results = processor.post_process_grounded_object_detection(
      outputs,
      inputs.input_ids,
      threshold=0.4,
      text_threshold=0.3,
      target_sizes=[image.size[::-1]]
  )

  result = results[0]
  if debug:
    for box, score, labels in zip(result["boxes"], result["scores"], result["labels"]):
        box = [round(x, 2) for x in box.tolist()]
        print(f"Detected {labels} with confidence {round(score.item(), 3)} at location {box}")

    # Display image with boxes
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    for box, score, label in zip(result["boxes"], result["scores"], result["labels"]):
      xmin, ymin, xmax, ymax = box.tolist()
      width, height = xmax - xmin, ymax - ymin
      rect = patches.Rectangle((xmin, ymin), width, height, linewidth=2, edgecolor='red', facecolor='none')
      ax.add_patch(rect)
      plt.text(xmin, ymin, f"{label}: {round(score.item(), 2)}",
              color='white', verticalalignment='top',
              bbox={'color': 'red', 'pad': 0})
    plt.axis('off')
    plt.show()
  return result

# OpenCV

In [ ]:
def auto_rotate(image_path):
    image = cv2.imread(image_path)
    osd = pytesseract.image_to_osd(image)
    rotate = int(osd.split("Rotate:")[1].split("\n")[0])

    if rotate == 90:
        cv2.imwrite(image_path,
                    cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE))
    elif rotate == 180:
        cv2.imwrite(image_path,
                    cv2.rotate(image, cv2.ROTATE_180))
    elif rotate == 270:
        cv2.imwrite(image_path,
                    cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE))


In [ ]:
def extract_text(image_path, debug = False):
  if debug:
    print("Finding text labels...")
  boxes = grounding_dino(image_path, [["Text Labels"]], debug)
  if debug:
    print("Extracting text...")
  labels = []
  for bbox in boxes["boxes"]:
    xmin, ymin, xmax, ymax = bbox.tolist()
    img = cv2.imread(image_path)
    cropped_img = img[int(ymin):int(ymax), int(xmin):int(xmax)]
    inverted_img = cv2.bitwise_not(cropped_img)
    text = pytesseract.image_to_string(inverted_img).rstrip()
    if debug:
      print("Raw: ", text.replace("\n", "\\n"))
    parts = text.split('\n')
    if debug:
      print("No of parts: ", len(parts))
    text = parts[1] if len(parts) > 1 else parts[0]
    clean_text = "".join(char for char in text if char.isalpha() or char == " ")
    clean_text = clean_text.strip()
    if debug:
      print("Cleaned: ", clean_text)
    labels.append({clean_text, bbox})
  return labels

In [ ]:
def remove_pans(image_path, debug = False):
  img = cv2.imread(image_path)

  hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
  h, s, v = cv2.split(hsv)

  mask = s < 30  # low saturation = gray / metal
  img[mask] = [255, 255, 255]

  if debug:
    cv2_imshow(img)

  return img

In [ ]:
def safe_crop(image, x1, y1, x2, y2):
    h, w = image.shape[:2]
    return image[
        max(0, y1) : min(h, y2),
        max(0, x1) : min(w, x2)
    ].copy()

In [ ]:
def extract_patches(img, name, debug=False, num_patches=5,
                    patch_size=(224, 224), strategy="grid"):
    """
    Extract patches from an image for OpenCV classification.

    Args:
        img: Path to source image
        output_dir: Where to save patches
        num_patches: How many patches to extract (4-5 recommended)
        patch_size: (width, height) of each patch
        strategy: "grid" for evenly spaced, "random" for random positions
    """
    output_dir = f'{base_path}/{name}/'
    os.makedirs(output_dir, exist_ok=True)

    h, w = img.shape[:2]
    pw, ph = patch_size
    patches = []

    # grid
    cols = int(np.ceil(np.sqrt(num_patches)))
    rows = int(np.ceil(num_patches / cols))
    x_steps = np.linspace(0, w - pw, cols, dtype=int)
    y_steps = np.linspace(0, h - ph, rows, dtype=int)
    coords = [(x, y) for y in y_steps for x in x_steps][:num_patches]

    # random
    np.random.seed(42)  # Remove for more options
    coords += ([
        (np.random.randint(0, max(1, w - pw)),
          np.random.randint(0, max(1, h - ph)))
        for _ in range(num_patches)
    ])

    # center & corners
    margin = 20
    coords += ([
        (w // 2 - pw // 2, h // 2 - ph // 2),
        (margin, margin),
        (w - pw - margin, margin),
        (margin, h - ph - margin),
        (w - pw - margin, h - ph - margin),
    ][:num_patches])

    if debug:
      print(coords)

    # Crop and save each patch
    existing = [f for f in os.listdir(output_dir) if f.startswith("patch_") and f.endswith(".jpg")]
    start_index = len(existing) + 1
    for i, (x, y) in enumerate(coords):
        patch = img[y:y+ph, x:x+pw]
        if patch.shape[:2] != (ph, pw):
            patch = cv2.resize(patch, (pw, ph))

        filename = os.path.join(output_dir, f"patch_{start_index + i:02d}.jpg")
        cv2.imwrite(filename, patch)
        patches.append(patch)
        if debug:
          print(f"Saved {filename} — crop at ({x}, {y})")

In [ ]:
def handle_image(image_path, debug = False):
  auto_rotate(image_path)
  food_boxes = grounding_dino(image_path, [["food"]])

  for bbox in food_boxes["boxes"]:
    xmin, ymin, xmax, ymax = bbox.tolist()
    img = cv2.imread(image_path)
    cropped_img = img[int(ymin):int(ymax), int(xmin):int(xmax)]
    temp_image_path = 'temp.jpg'
    cv2.imwrite(temp_image_path, cropped_img)
    cleaned_img = remove_pans(temp_image_path, debug)
    name = ""
    try:
      label = extract_text(temp_image_path, debug)
      name = list(label[0])[1]
      if name == "":
        name = "Unknown"
    except Exception as e:
      print("Error: ", e)
      print("Image name: ", image_path)
      name = "Unknown"
      cv2_imshow(cleaned_img)
    cv2.imwrite(f"{base_path}/uncropped/{name}.jpg", cleaned_img)
    os.remove(temp_image_path)
    extract_patches(cleaned_img, name, debug)
    if debug:
      cv2_imshow(cleaned_img)
      print(labels)

In [ ]:
jpg_files = [f for f in os.listdir(f'{base_path}/') if f.lower().endswith('.jpg')]
for image in jpg_files:
  # try:
    handle_image(f'{base_path}/{image}', False)
  # except Exception as e:
  #   print("Error: ", e)
  #   print("Image name: ", image)

TesseractError: (1, 'Tesseract Open Source OCR Engine v4.1.1 with Leptonica Warning: Invalid resolution 0 dpi. Using 70 instead. Estimating resolution as 183 Too few characters. Skipping this page Warning. Invalid resolution 0 dpi. Using 70 instead. Too few characters. Skipping this page Error during processing.')

In [ ]:
import os
from google.colab import files

# 1. Automatically get names of all visible folders (ignoring system folders)
ignored = ['sample_data', '.config', '.ipynb_checkpoints']
folders = [d for d in os.listdir('.') if os.path.isdir(d) and d not in ignored and not d.startswith('.')]

if folders:
    # 2. Join folder names into a string and zip them
    folder_string = " ".join([f'"{f}"' for f in folders])
    os.system(f'zip -r all_folders.zip {folder_string}')

    # 3. Trigger the download
    print(f"Zipping and downloading: {folders}")
    files.download('all_folders.zip')
else:
    print("No custom folders found to download.")

Zipping and downloading: ['Pork Head Sari Sari', "tensor([1.4307e-02, 3.7077e-02, 1.6601e+02, 1.3004e+02], device='cuda:0')", 'Fish Paksiw', "tensor([   2.8778,    1.6310, 1102.1254,  319.9696], device='cuda:0')", "tensor([  1.6663,   3.5449, 483.2805, 409.3374], device='cuda:0')", "tensor([ 9.5309e-01, -3.3200e-04,  4.2527e+02,  5.5699e+02], device='cuda:0')", "tensor([ 246.9357,  988.0944,  488.3756, 1080.5248], device='cuda:0')", "tensor([   5.1878,    2.1430, 1045.7013,  649.9329], device='cuda:0')", 'Gata', 'Fried Buwad Bolinao', "tensor([  3.8064,   4.0549, 657.4113, 928.8574], device='cuda:0')", "tensor([7.5918e-01, 1.8553e-01, 1.4123e+02, 6.4793e+02], device='cuda:0')", "tensor([326.2497, 516.1947, 384.5801, 813.2877], device='cuda:0')", 'Chicken Caldereta', 'Hotdog', "tensor([177.4460, 393.5236, 552.4514, 484.8544], device='cuda:0')", 'Pork Belly Humt', "tensor([   4.2588,    8.7635,  642.3972, 1030.3124], device='cuda:0')", "tensor([  2.7019,   5.2515, 610.6614, 506.6707], de

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>